# IPL Prematch Winner Prediction — v3
### Builds on your exact working pipeline (64% LR baseline) → target 75%+
**Strategy:** Same 80/20 chronological split + OHE that worked, but with richer leakage-safe features + tuned XGBoost

In [ ]:
# ============================================================
# CELL 1: Imports
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, GradientBoostingClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, ConfusionMatrixDisplay)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

print('All imports OK!')

In [ ]:
# ============================================================
# CELL 2: Load & build match-level dataframe
# (same as your existing notebook)
# ============================================================
df = pd.read_csv('../data/IPL.csv', low_memory=False)

match_columns = ['match_id', 'batting_team', 'bowling_team', 'toss_winner',
                 'toss_decision', 'venue', 'city', 'match_won_by',
                 'day', 'month', 'year', 'stage']
match_columns = [c for c in match_columns if c in df.columns]

match_df = df[match_columns].groupby('match_id').first().reset_index()
match_df = match_df.rename(columns={
    'match_won_by': 'winner',
    'batting_team': 'team1',
    'bowling_team': 'team2'
})

match_df = match_df.dropna(subset=['winner'])
match_df = match_df[match_df['winner'] != 'Unknown']

# Sort chronologically — same as your existing notebook
match_df = match_df.sort_values('match_id').reset_index(drop=True)

print('Match-level shape:', match_df.shape)
print(match_df[['match_id','team1','team2','winner','year']].head())

In [ ]:
# ============================================================
# CELL 3: Standardize venue names
# (same as your existing notebook)
# ============================================================
venue_map = {
    'Arun Jaitley Stadium, Delhi': 'Arun Jaitley Stadium',
    'Brabourne Stadium, Mumbai': 'Brabourne Stadium',
    'M Chinnaswamy Stadium, Bengaluru': 'M Chinnaswamy Stadium',
    'M.Chinnaswamy Stadium': 'M Chinnaswamy Stadium',
    'MA Chidambaram Stadium, Chepauk': 'MA Chidambaram Stadium',
    'MA Chidambaram Stadium, Chepauk, Chennai': 'MA Chidambaram Stadium',
    'Punjab Cricket Association IS Bindra Stadium, Mohali': 'PCA IS Bindra Stadium',
    'Punjab Cricket Association IS Bindra Stadium, Mohali, Chandigarh': 'PCA IS Bindra Stadium',
    'Punjab Cricket Association Stadium, Mohali': 'PCA IS Bindra Stadium',
    'Punjab Cricket Association IS Bindra Stadium': 'PCA IS Bindra Stadium',
    'Rajiv Gandhi International Stadium, Uppal': 'Rajiv Gandhi International Stadium',
    'Rajiv Gandhi International Stadium, Uppal, Hyderabad': 'Rajiv Gandhi International Stadium',
    'Sawai Mansingh Stadium, Jaipur': 'Sawai Mansingh Stadium',
    'Wankhede Stadium, Mumbai': 'Wankhede Stadium',
    'Dr DY Patil Sports Academy, Mumbai': 'Dr DY Patil Sports Academy',
    'Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium, Visakhapatnam': 'ACA-VDCA Stadium',
    'Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium': 'ACA-VDCA Stadium',
    'Himachal Pradesh Cricket Association Stadium, Dharamsala': 'HPCA Stadium',
    'Himachal Pradesh Cricket Association Stadium': 'HPCA Stadium',
    'Maharashtra Cricket Association Stadium, Pune': 'MCA Stadium',
    'Maharashtra Cricket Association Stadium': 'MCA Stadium',
    'Narendra Modi Stadium, Ahmedabad': 'Narendra Modi Stadium',
    'Sardar Patel Stadium, Motera': 'Narendra Modi Stadium',
    'Zayed Cricket Stadium, Abu Dhabi': 'Sheikh Zayed Stadium',
    'Eden Gardens, Kolkata': 'Eden Gardens'
}
match_df['venue'] = match_df['venue'].replace(venue_map)
print('Unique venues:', match_df['venue'].nunique())

In [ ]:
# ============================================================
# CELL 4: ALL feature engineering — leakage-safe sequential loop
# Records features BEFORE updating history each match
# ============================================================

team_wins    = {}
team_played  = {}
team_history = {}   # list of 1/0 results
h2h          = {}   # head-to-head {frozenset: {played, team: wins}}
venue_w      = {}   # (team, venue) -> wins
venue_p      = {}   # (team, venue) -> played
season_w     = {}   # (team, season) -> wins
season_p     = {}   # (team, season) -> played
elo          = {}   # Elo ratings (start 1500)
K            = 20

cols = [
    'team1_win_pct', 'team2_win_pct',
    'h2h_win_pct',
    'team1_last7_form', 'team2_last7_form',
    'team1_last5_form', 'team2_last5_form',
    'team1_last3_form', 'team2_last3_form',
    'team1_venue_win_pct', 'team2_venue_win_pct',
    'team1_season_form', 'team2_season_form',
    'elo_diff',
]
out = {c: [] for c in cols}

for _, row in match_df.iterrows():
    t1     = row['team1']
    t2     = row['team2']
    w      = row['winner']
    v      = row['venue']
    s      = row['year']

    for t in [t1, t2]:
        team_wins.setdefault(t, 0)
        team_played.setdefault(t, 0)
        team_history.setdefault(t, [])
        venue_w.setdefault((t, v), 0)
        venue_p.setdefault((t, v), 0)
        season_w.setdefault((t, s), 0)
        season_p.setdefault((t, s), 0)
        elo.setdefault(t, 1500)

    pair = frozenset([t1, t2])
    h2h.setdefault(pair, {'played': 0, t1: 0, t2: 0})

    # --- Record features BEFORE updating ---
    out['team1_win_pct'].append(team_wins[t1] / team_played[t1] if team_played[t1] else 0.5)
    out['team2_win_pct'].append(team_wins[t2] / team_played[t2] if team_played[t2] else 0.5)

    hp = h2h[pair]['played']
    out['h2h_win_pct'].append(h2h[pair].get(t1, 0) / hp if hp else 0.5)

    h1 = team_history[t1]; h2_ = team_history[t2]
    def form(hist, n): return sum(hist[-n:]) / len(hist[-n:]) if hist[-n:] else 0.5
    out['team1_last7_form'].append(form(h1, 7));  out['team2_last7_form'].append(form(h2_, 7))
    out['team1_last5_form'].append(form(h1, 5));  out['team2_last5_form'].append(form(h2_, 5))
    out['team1_last3_form'].append(form(h1, 3));  out['team2_last3_form'].append(form(h2_, 3))

    vp1 = venue_p[(t1, v)]; vp2 = venue_p[(t2, v)]
    out['team1_venue_win_pct'].append(venue_w[(t1, v)] / vp1 if vp1 else 0.5)
    out['team2_venue_win_pct'].append(venue_w[(t2, v)] / vp2 if vp2 else 0.5)

    sp1 = season_p[(t1, s)]; sp2 = season_p[(t2, s)]
    out['team1_season_form'].append(season_w[(t1, s)] / sp1 if sp1 else 0.5)
    out['team2_season_form'].append(season_w[(t2, s)] / sp2 if sp2 else 0.5)

    out['elo_diff'].append(elo[t1] - elo[t2])

    # --- Update AFTER recording ---
    t1_won = (w == t1)
    team_wins[t1]   += int(t1_won);   team_wins[t2]   += int(not t1_won)
    team_played[t1] += 1;             team_played[t2] += 1
    team_history[t1].append(int(t1_won));  team_history[t2].append(int(not t1_won))
    h2h[pair]['played'] += 1
    h2h[pair][w] = h2h[pair].get(w, 0) + 1
    venue_w[(t1, v)] += int(t1_won);  venue_w[(t2, v)] += int(not t1_won)
    venue_p[(t1, v)] += 1;            venue_p[(t2, v)] += 1
    season_w[(t1, s)] += int(t1_won); season_w[(t2, s)] += int(not t1_won)
    season_p[(t1, s)] += 1;           season_p[(t2, s)] += 1
    r1, r2 = elo[t1], elo[t2]
    e1 = 1 / (1 + 10 ** ((r2 - r1) / 400))
    s1, s2 = (1, 0) if t1_won else (0, 1)
    elo[t1] = r1 + K * (s1 - e1)
    elo[t2] = r2 + K * (s2 - (1 - e1))

for c, v in out.items():
    match_df[c] = v

print('Sequential features done. Shape:', match_df.shape)

In [ ]:
# ============================================================
# CELL 5: Remaining features (home, toss, chasing, venue chasing, diffs)
# ============================================================

# Home grounds
home_grounds = {
    'Chennai Super Kings': 'MA Chidambaram Stadium',
    'Mumbai Indians': 'Wankhede Stadium',
    'Royal Challengers Bangalore': 'M Chinnaswamy Stadium',
    'Royal Challengers Bengaluru': 'M Chinnaswamy Stadium',
    'Kolkata Knight Riders': 'Eden Gardens',
    'Delhi Capitals': 'Arun Jaitley Stadium',
    'Delhi Daredevils': 'Feroz Shah Kotla',
    'Punjab Kings': 'PCA IS Bindra Stadium',
    'Kings XI Punjab': 'PCA IS Bindra Stadium',
    'Rajasthan Royals': 'Sawai Mansingh Stadium',
    'Sunrisers Hyderabad': 'Rajiv Gandhi International Stadium',
    'Deccan Chargers': 'Rajiv Gandhi International Stadium',
    'Lucknow Super Giants': 'Bharat Ratna Shri Atal Bihari Vajpayee Ekana Cricket Stadium, Lucknow',
    'Gujarat Titans': 'Narendra Modi Stadium',
    'Rising Pune Supergiants': 'MCA Stadium',
    'Rising Pune Supergiant': 'MCA Stadium',
}
match_df['team1_home'] = match_df.apply(lambda r: int(home_grounds.get(r['team1']) == r['venue']), axis=1)
match_df['team2_home'] = match_df.apply(lambda r: int(home_grounds.get(r['team2']) == r['venue']), axis=1)

# Toss
match_df['team1_won_toss']      = (match_df['toss_winner'] == match_df['team1']).astype(int)
match_df['toss_decision_field'] = (match_df['toss_decision'] == 'field').astype(int)

# Chasing team
def get_chasing(row):
    if row['toss_decision'] == 'field':
        return row['toss_winner']
    return row['team1'] if row['toss_winner'] != row['team1'] else row['team2']

match_df['chasing_team'] = match_df.apply(get_chasing, axis=1)
match_df['team1_chasing'] = (match_df['chasing_team'] == match_df['team1']).astype(int)

# Venue chasing win rate (global, not sequential — same as your original notebook)
match_df['chasing_won'] = (match_df['winner'] == match_df['chasing_team']).astype(int)
venue_chase_rate = match_df.groupby('venue')['chasing_won'].mean()
match_df['venue_chasing_rate'] = match_df['venue'].map(venue_chase_rate)

# Playoff flag
if 'stage' in match_df.columns:
    keywords = ['final', 'qualifier', 'eliminator', 'semi', 'playoff']
    match_df['is_playoff'] = match_df['stage'].str.lower().str.contains('|'.join(keywords), na=False).astype(int)
else:
    match_df['is_playoff'] = 0

# Difference features (team1 - team2)
match_df['win_pct_diff']       = match_df['team1_win_pct']       - match_df['team2_win_pct']
match_df['form7_diff']         = match_df['team1_last7_form']    - match_df['team2_last7_form']
match_df['form5_diff']         = match_df['team1_last5_form']    - match_df['team2_last5_form']
match_df['form3_diff']         = match_df['team1_last3_form']    - match_df['team2_last3_form']
match_df['venue_win_pct_diff'] = match_df['team1_venue_win_pct'] - match_df['team2_venue_win_pct']
match_df['season_form_diff']   = match_df['team1_season_form']   - match_df['team2_season_form']
match_df['home_advantage']     = match_df['team1_home']          - match_df['team2_home']

# Binary target
match_df['team1_win'] = (match_df['winner'] == match_df['team1']).astype(int)

print('All features ready!')
print('Class balance:', match_df['team1_win'].value_counts().to_dict())
print('Total features added:', match_df.shape[1])

In [ ]:
# ============================================================
# CELL 6: Define features & EXACT same split as your notebook
# 80/20 chronological split on sorted match_id
# ============================================================

CATEGORICAL = ['team1', 'team2', 'venue', 'toss_winner', 'toss_decision']

NUMERIC = [
    # Original features you already had
    'team1_win_pct', 'team2_win_pct',
    'h2h_win_pct',
    'team1_venue_win_pct',
    'team1_home', 'team2_home',
    'venue_chasing_rate',
    'team1_won_toss', 'toss_decision_field', 'team1_chasing',
    'team1_last7_form', 'team2_last7_form',
    # New features
    'team1_last5_form', 'team2_last5_form',
    'team1_last3_form', 'team2_last3_form',
    'team2_venue_win_pct',
    'team1_season_form', 'team2_season_form',
    'elo_diff',
    'win_pct_diff', 'form7_diff', 'form5_diff', 'form3_diff',
    'venue_win_pct_diff', 'season_form_diff', 'home_advantage',
    'is_playoff',
]

ALL_FEATURES = CATEGORICAL + NUMERIC
TARGET = 'team1_win'

X = match_df[ALL_FEATURES]
y = match_df[TARGET]

# Same 80/20 chronological split as your notebook
split_idx = int(len(match_df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f'Train: {len(X_train)} matches')
print(f'Test : {len(X_test)} matches')
print(f'Total features: {len(ALL_FEATURES)} ({len(CATEGORICAL)} cat + {len(NUMERIC)} numeric)')

In [ ]:
# ============================================================
# CELL 7: Preprocessor (OHE for categorical — same as your notebook)
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CATEGORICAL)
    ],
    remainder='passthrough'
)

print('Preprocessor ready!')

In [ ]:
# ============================================================
# CELL 8: BASELINE — Logistic Regression (reproduce your 64%)
# ============================================================

lr_pipe = Pipeline([
    ('pre', preprocessor),
    ('clf', LogisticRegression(max_iter=2000, random_state=42))
])
lr_pipe.fit(X_train, y_train)
y_pred_lr = lr_pipe.predict(X_test)

print('=== Logistic Regression (Baseline) ===')
print(f'Accuracy : {accuracy_score(y_test, y_pred_lr):.4f}')
print(f'Precision: {precision_score(y_test, y_pred_lr):.4f}')
print(f'Recall   : {recall_score(y_test, y_pred_lr):.4f}')
print(f'F1 Score : {f1_score(y_test, y_pred_lr):.4f}')

In [ ]:
# ============================================================
# CELL 9: Random Forest
# ============================================================

rf_pipe = Pipeline([
    ('pre', preprocessor),
    ('clf', RandomForestClassifier(
        n_estimators=500,
        max_depth=8,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1
    ))
])
rf_pipe.fit(X_train, y_train)
y_pred_rf = rf_pipe.predict(X_test)

print('=== Random Forest ===')
print(f'Accuracy : {accuracy_score(y_test, y_pred_rf):.4f}')
print(f'Precision: {precision_score(y_test, y_pred_rf):.4f}')
print(f'Recall   : {recall_score(y_test, y_pred_rf):.4f}')
print(f'F1 Score : {f1_score(y_test, y_pred_rf):.4f}')

In [ ]:
# ============================================================
# CELL 10: XGBoost — properly tuned for small tabular dataset
# ============================================================

# Fit preprocessor separately so XGBoost gets numpy array
preprocessor_xgb = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CATEGORICAL)
    ],
    remainder='passthrough'
)
X_train_xgb = preprocessor_xgb.fit_transform(X_train)
X_test_xgb  = preprocessor_xgb.transform(X_test)

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=3,          # shallow — avoids overfitting on small dataset
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,   # important for small datasets
    reg_alpha=0.1,        # L1 regularization
    reg_lambda=1.0,       # L2 regularization
    eval_metric='logloss',
    random_state=42,
    verbosity=0
)
xgb.fit(X_train_xgb, y_train)
y_pred_xgb = xgb.predict(X_test_xgb)

print('=== XGBoost ===')
print(f'Accuracy : {accuracy_score(y_test, y_pred_xgb):.4f}')
print(f'Precision: {precision_score(y_test, y_pred_xgb):.4f}')
print(f'Recall   : {recall_score(y_test, y_pred_xgb):.4f}')
print(f'F1 Score : {f1_score(y_test, y_pred_xgb):.4f}')
print(f'ROC-AUC  : {roc_auc_score(y_test, xgb.predict_proba(X_test_xgb)[:,1]):.4f}')

In [ ]:
# ============================================================
# CELL 11: LightGBM
# ============================================================

lgbm = LGBMClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_samples=10,
    reg_alpha=0.1,
    random_state=42,
    verbose=-1
)
lgbm.fit(X_train_xgb, y_train)
y_pred_lgbm = lgbm.predict(X_test_xgb)

print('=== LightGBM ===')
print(f'Accuracy : {accuracy_score(y_test, y_pred_lgbm):.4f}')
print(f'Precision: {precision_score(y_test, y_pred_lgbm):.4f}')
print(f'Recall   : {recall_score(y_test, y_pred_lgbm):.4f}')
print(f'F1 Score : {f1_score(y_test, y_pred_lgbm):.4f}')
print(f'ROC-AUC  : {roc_auc_score(y_test, lgbm.predict_proba(X_test_xgb)[:,1]):.4f}')

In [ ]:
# ============================================================
# CELL 12: Soft Voting Ensemble (LR + RF + XGB + LGBM)
# Probability averaging — best strategy for small datasets
# ============================================================

from sklearn.calibration import CalibratedClassifierCV

# Get probabilities from all models
p_lr   = lr_pipe.predict_proba(X_test)[:, 1]
p_rf   = rf_pipe.predict_proba(X_test)[:, 1]
p_xgb  = xgb.predict_proba(X_test_xgb)[:, 1]
p_lgbm = lgbm.predict_proba(X_test_xgb)[:, 1]

# Try different ensemble weights
best_acc = 0
best_w = None
best_preds = None

for w_lr, w_rf, w_xgb, w_lgbm in [
    (1, 1, 1, 1),
    (2, 1, 1, 1),
    (1, 2, 1, 1),
    (1, 1, 2, 1),
    (1, 1, 1, 2),
    (2, 1, 2, 2),
    (1, 2, 2, 2),
    (2, 2, 1, 1),
    (3, 1, 2, 2),
]:
    total = w_lr + w_rf + w_xgb + w_lgbm
    p_ens = (w_lr*p_lr + w_rf*p_rf + w_xgb*p_xgb + w_lgbm*p_lgbm) / total
    y_ens = (p_ens >= 0.5).astype(int)
    acc = accuracy_score(y_test, y_ens)
    if acc > best_acc:
        best_acc = acc
        best_w = (w_lr, w_rf, w_xgb, w_lgbm)
        best_preds = y_ens
        best_proba = p_ens

print(f'Best ensemble weights: LR={best_w[0]}, RF={best_w[1]}, XGB={best_w[2]}, LGBM={best_w[3]}')
print()
print('=== Best Ensemble ===')
print(f'Accuracy : {accuracy_score(y_test, best_preds):.4f}')
print(f'Precision: {precision_score(y_test, best_preds):.4f}')
print(f'Recall   : {recall_score(y_test, best_preds):.4f}')
print(f'F1 Score : {f1_score(y_test, best_preds):.4f}')
print(f'ROC-AUC  : {roc_auc_score(y_test, best_proba):.4f}')

In [ ]:
# ============================================================
# CELL 13: Full Comparison Table
# ============================================================

results = [
    {'Model': 'Logistic Regression',
     'Accuracy': accuracy_score(y_test, y_pred_lr),
     'Precision': precision_score(y_test, y_pred_lr),
     'Recall': recall_score(y_test, y_pred_lr),
     'F1': f1_score(y_test, y_pred_lr),
     'ROC-AUC': roc_auc_score(y_test, lr_pipe.predict_proba(X_test)[:,1])},
    {'Model': 'Random Forest',
     'Accuracy': accuracy_score(y_test, y_pred_rf),
     'Precision': precision_score(y_test, y_pred_rf),
     'Recall': recall_score(y_test, y_pred_rf),
     'F1': f1_score(y_test, y_pred_rf),
     'ROC-AUC': roc_auc_score(y_test, rf_pipe.predict_proba(X_test)[:,1])},
    {'Model': 'XGBoost',
     'Accuracy': accuracy_score(y_test, y_pred_xgb),
     'Precision': precision_score(y_test, y_pred_xgb),
     'Recall': recall_score(y_test, y_pred_xgb),
     'F1': f1_score(y_test, y_pred_xgb),
     'ROC-AUC': roc_auc_score(y_test, xgb.predict_proba(X_test_xgb)[:,1])},
    {'Model': 'LightGBM',
     'Accuracy': accuracy_score(y_test, y_pred_lgbm),
     'Precision': precision_score(y_test, y_pred_lgbm),
     'Recall': recall_score(y_test, y_pred_lgbm),
     'F1': f1_score(y_test, y_pred_lgbm),
     'ROC-AUC': roc_auc_score(y_test, lgbm.predict_proba(X_test_xgb)[:,1])},
    {'Model': f'Ensemble (best weights)',
     'Accuracy': accuracy_score(y_test, best_preds),
     'Precision': precision_score(y_test, best_preds),
     'Recall': recall_score(y_test, best_preds),
     'F1': f1_score(y_test, best_preds),
     'ROC-AUC': roc_auc_score(y_test, best_proba)},
]

res_df = pd.DataFrame(results).sort_values('Accuracy', ascending=False)
res_df = res_df.round(4)
print(res_df.to_string(index=False))

In [ ]:
# ============================================================
# CELL 14: XGBoost Feature Importance
# ============================================================

feature_names = (
    list(preprocessor_xgb.named_transformers_['cat'].get_feature_names_out(CATEGORICAL))
    + NUMERIC
)

imp = pd.Series(xgb.feature_importances_, index=feature_names)
# Show only numeric features for clarity (OHE columns are many)
imp_numeric = imp[NUMERIC].sort_values(ascending=True)

plt.figure(figsize=(9, 8))
imp_numeric.plot(kind='barh', color='steelblue')
plt.title('XGBoost — Numeric Feature Importances')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

print('\nTop 10 features overall:')
print(imp.sort_values(ascending=False).head(10))

In [ ]:
# ============================================================
# CELL 15: Confusion Matrix of best model
# ============================================================

best_row = res_df.iloc[0]
print(f"Best model: {best_row['Model']} | Accuracy: {best_row['Accuracy']}")

ConfusionMatrixDisplay.from_predictions(
    y_test, best_preds,
    display_labels=['Team2 Wins', 'Team1 Wins']
)
plt.title(f'Confusion Matrix — {best_row["Model"]}')
plt.show()